In [19]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import pyspark.sql.functions as F


In [20]:
spark = configure_spark_with_delta_pip(
    SparkSession.builder
    .appName("Fligh Delays and Cancellations")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")    
).getOrCreate()

# Read Data Files

In [21]:
DIRECTORY = "./dataset"

In [22]:
airlines = spark.read.csv(f"{DIRECTORY}/airlines.csv", header=True, inferSchema=True)

In [23]:
airports = spark.read.csv(f"{DIRECTORY}/airports.csv", header=True, inferSchema=True)

In [24]:
flights = spark.read.csv(f"{DIRECTORY}/flights.csv", header=True, inferSchema=True)

Explore Inferred Schema and Create Explicit Schema (if needed)

In [25]:
airlines.printSchema()

root
 |-- IATA_CODE: string (nullable = true)
 |-- AIRLINE: string (nullable = true)



In [26]:
airlines.show(5, False)

+---------+----------------------+
|IATA_CODE|AIRLINE               |
+---------+----------------------+
|UA       |United Air Lines Inc. |
|AA       |American Airlines Inc.|
|US       |US Airways Inc.       |
|F9       |Frontier Airlines Inc.|
|B6       |JetBlue Airways       |
+---------+----------------------+
only showing top 5 rows


In [27]:
airports.printSchema()

root
 |-- IATA_CODE: string (nullable = true)
 |-- AIRPORT: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- COUNTRY: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)



In [28]:
airports.show(5, False)

+---------+-----------------------------------+-----------+-----+-------+--------+----------+
|IATA_CODE|AIRPORT                            |CITY       |STATE|COUNTRY|LATITUDE|LONGITUDE |
+---------+-----------------------------------+-----------+-----+-------+--------+----------+
|ABE      |Lehigh Valley International Airport|Allentown  |PA   |USA    |40.65236|-75.4404  |
|ABI      |Abilene Regional Airport           |Abilene    |TX   |USA    |32.41132|-99.6819  |
|ABQ      |Albuquerque International Sunport  |Albuquerque|NM   |USA    |35.04022|-106.60919|
|ABR      |Aberdeen Regional Airport          |Aberdeen   |SD   |USA    |45.44906|-98.42183 |
|ABY      |Southwest Georgia Regional Airport |Albany     |GA   |USA    |31.53552|-84.19447 |
+---------+-----------------------------------+-----------+-----+-------+--------+----------+
only showing top 5 rows


In [29]:
flights.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- FLIGHT_NUMBER: integer (nullable = true)
 |-- TAIL_NUMBER: string (nullable = true)
 |-- ORIGIN_AIRPORT: string (nullable = true)
 |-- DESTINATION_AIRPORT: string (nullable = true)
 |-- SCHEDULED_DEPARTURE: integer (nullable = true)
 |-- DEPARTURE_TIME: integer (nullable = true)
 |-- DEPARTURE_DELAY: integer (nullable = true)
 |-- TAXI_OUT: integer (nullable = true)
 |-- WHEELS_OFF: integer (nullable = true)
 |-- SCHEDULED_TIME: integer (nullable = true)
 |-- ELAPSED_TIME: integer (nullable = true)
 |-- AIR_TIME: integer (nullable = true)
 |-- DISTANCE: integer (nullable = true)
 |-- WHEELS_ON: integer (nullable = true)
 |-- TAXI_IN: integer (nullable = true)
 |-- SCHEDULED_ARRIVAL: integer (nullable = true)
 |-- ARRIVAL_TIME: integer (nullable = true)
 |-- ARRIVAL_DELAY: integer (null

In [30]:
flights.show(5, False)

+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+--------+---------+-------+-----------------+------------+-------------+--------+---------+-------------------+----------------+--------------+-------------+-------------------+-------------+
|YEAR|MONTH|DAY|DAY_OF_WEEK|AIRLINE|FLIGHT_NUMBER|TAIL_NUMBER|ORIGIN_AIRPORT|DESTINATION_AIRPORT|SCHEDULED_DEPARTURE|DEPARTURE_TIME|DEPARTURE_DELAY|TAXI_OUT|WHEELS_OFF|SCHEDULED_TIME|ELAPSED_TIME|AIR_TIME|DISTANCE|WHEELS_ON|TAXI_IN|SCHEDULED_ARRIVAL|ARRIVAL_TIME|ARRIVAL_DELAY|DIVERTED|CANCELLED|CANCELLATION_REASON|AIR_SYSTEM_DELAY|SECURITY_DELAY|AIRLINE_DELAY|LATE_AIRCRAFT_DELAY|WEATHER_DELAY|
+----+-----+---+-----------+-------+-------------+-----------+--------------+-------------------+-------------------+--------------+---------------+--------+----------+--------------+------------+--------+-

# Save Raw Data to Delta Table Bronze Layer

In [31]:
(airlines.write
    .format("delta")
    .mode("overwrite")
    .save("./medallion/bronze/airlines_bronze")

)

In [32]:
(airports.write
    .format("delta")
    .mode("overwrite")
    .save("./medallion/bronze/airports_bronze")
)

In [33]:
(flights.write
    .format("delta")
    .mode("overwrite")
    .save("./medallion/bronze/flights_bronze")
)

26/03/14 19:54:14 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

# Read Delta Tables from Bronze Layer

In [36]:
airlines = spark.read.format("delta").load("./medallion/bronze/airlines_bronze")

In [38]:
airports = spark.read.format("delta").load("./medallion/bronze/airports_bronze")

In [40]:
fligths = spark.read.format("delta").load("./medallion/bronze/flights_bronze")

# Data Checks

In [47]:
airlines.show(14, False)

+---------+----------------------------+
|IATA_CODE|AIRLINE                     |
+---------+----------------------------+
|UA       |United Air Lines Inc.       |
|AA       |American Airlines Inc.      |
|US       |US Airways Inc.             |
|F9       |Frontier Airlines Inc.      |
|B6       |JetBlue Airways             |
|OO       |Skywest Airlines Inc.       |
|AS       |Alaska Airlines Inc.        |
|NK       |Spirit Air Lines            |
|WN       |Southwest Airlines Co.      |
|DL       |Delta Air Lines Inc.        |
|EV       |Atlantic Southeast Airlines |
|HA       |Hawaiian Airlines Inc.      |
|MQ       |American Eagle Airlines Inc.|
|VX       |Virgin America              |
+---------+----------------------------+



In [42]:
airlines.count()

14

In [48]:
airports.count()

322

In [52]:
airports.show(10, False)

+---------+-----------------------------------+-------------+-----+-------+--------+----------+
|IATA_CODE|AIRPORT                            |CITY         |STATE|COUNTRY|LATITUDE|LONGITUDE |
+---------+-----------------------------------+-------------+-----+-------+--------+----------+
|ABE      |Lehigh Valley International Airport|Allentown    |PA   |USA    |40.65236|-75.4404  |
|ABI      |Abilene Regional Airport           |Abilene      |TX   |USA    |32.41132|-99.6819  |
|ABQ      |Albuquerque International Sunport  |Albuquerque  |NM   |USA    |35.04022|-106.60919|
|ABR      |Aberdeen Regional Airport          |Aberdeen     |SD   |USA    |45.44906|-98.42183 |
|ABY      |Southwest Georgia Regional Airport |Albany       |GA   |USA    |31.53552|-84.19447 |
|ACK      |Nantucket Memorial Airport         |Nantucket    |MA   |USA    |41.25305|-70.06018 |
|ACT      |Waco Regional Airport              |Waco         |TX   |USA    |31.61129|-97.23052 |
|ACV      |Arcata Airport               

In [57]:
[c.isNull()for c in airports.columns]

AttributeError: 'str' object has no attribute 'isNull'